# 🛡️ AI vs Real Image Detection - AIForensicNet V4 (State-of-the-Art Forensic Architecture)

Notebook ini merupakan implementasi dan penyempurnaan menyeluruh dari arsitektur model kustom **AIForensicNet V4** yang dilatih menggunakan dua dataset mutakhir berstandar global:
1. **KarmaLeo/Xiz9Dataset**: Generator AI mutakhir (*FLUX.1, Midjourney v6, DALL-E 3, SDXL, GPT-Image-1, Grok-2*) vs Foto & Karya Seni Asli (*Pexels, WikiArt, LAION*).
2. **semi-truths/Semi-Truths (NeurIPS 2024)**: Manipulasi lokal (*AI Inpainting & Prompt-to-Prompt localized edits*) vs Pemandangan Otentik (*ADE20K*).

### 🌟 Inovasi & Keunggulan AIForensicNet V4:
1. **Tri-Stream Architecture (Fusi 3 Ranah Forensik)**:
   - **Aliran 1 (Multi-Scale Residual + NPR - 15 Kanal)**: Menggabungkan Laplacian 3x3, Laplacian 5x5, Difference of Gaussians (DoG 5x5), Sobel Magnitude, dan **NPR (Neighboring Pixel Relationships - CVPR 2024)** untuk mendeteksi artefak *transposed convolution / upsampling* khas GAN & Latent Diffusion Decoders.
   - **Aliran 2 (2D-FFT Spectral Branch - 32 Kanal)**: Mengekstraksi anomali simetri radial pada spektrum Fourier frekuensi 2D yang tidak tampak pada domain spasial.
   - **Aliran 3 (Spatial RGB - 32 Kanal)**: Mengekstraksi koherensi semantik visual, pencahayaan alami, dan tekstur objek asli.
   - **Tri-Stream Projection**: Fusi 96 kanal diproyeksikan secara harmonis ke 64 kanal.
2. **Native-Resolution Random Patch Cropping**:
   - Mengeliminasi proses downsampling global `Resize((224, 224))` yang merusak sinyal frekuensi tinggi mikro. Citra di-crop langsung dari resolusi asli (*native pixel resolution*) sehingga sidik jari sintesis piksel tetap utuh.
3. **Metodologi Akademis Emas (3-Way Stratified Split)**:
   - Membagi data secara ketat menjadi **80% Training**, **10% Validation** (seleksi checkpoint terbaik), dan **10% Held-Out Test Set** (dievaluasi strictly 1 kali di akhir).
4. **Multi-Stat Global Pooling (`MultiStatPool2d`)**:
   - Menggabungkan momen pertama (Mean aktivasi) dan momen kedua (Standard Deviation sebaran spasial energi derau) menjadi representasi 1024-dimensi.
5. **Regulasi Pelatihan Modern**:
   - Automatic Mixed Precision (AMP FP16), Label Smoothing (0.05), Gradient Clipping, dan Cosine Annealing dengan Linear Warmup.

In [ ]:
# [Opsional Colab] Pastikan library yang dibutuhkan terpasang
!pip install -q datasets torchvision scikit-learn pillow matplotlib tqdm pyarrow huggingface_hub


In [ ]:
import os
import sys
import time
import copy
import json
import random
import warnings
from io import BytesIO

from PIL import Image
# Cegah DecompressionBombWarning & Error pada citra resolusi tinggi (>89 Megapixels)
Image.MAX_IMAGE_PIXELS = None
warnings.simplefilter('ignore', Image.DecompressionBombWarning)
warnings.filterwarnings('ignore', message='.*Palette images with Transparency.*')
warnings.filterwarnings('ignore', message='.*xFormers is not available.*')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from datasets import load_dataset

# ------------------ PENGATURAN SEED & DEVICE ------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 70)
if torch.cuda.is_available():
    print(f"Device Utama : {device} ({torch.cuda.get_device_name(0)})")
    print(f"Total VRAM   : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print(f"Device Utama : {device} (Peringatan: Gunakan akselerasi GPU pada Colab)")
print('=' * 70)


## 📦 1. Pemuatan Dataset Xiz9 & Semi-Truths serta Pembagian Data (3-Way Split: 80% Train, 10% Val, 10% Test)
Untuk melatih model agar tangguh mendeteksi citra sintetis modern, kita memadukan dua dataset berstandar SOTA:
1. **KarmaLeo/Xiz9Dataset**: Mencakup generator difusi & autoregresif mutakhir (*FLUX.1, Midjourney v6, DALL-E 3, SDXL, GPT-Image-1, Grok-2*) berhadapan dengan foto/karya seni asli (*Pexels, WikiArt, LAION*).
2. **semi-truths/Semi-Truths (NeurIPS 2024)**: Tolok ukur mitigasi manipulasi lokal (*AI Inpainting & Prompt-to-Prompt localized editing*) berhadapan dengan pemandangan asli (*ADE20K*).

Dataset dibagi secara berstrata (*stratified*) menjadi 3 himpunan yang saling bebas (*disjoint*):
- **Train Set (80%)**: Digunakan untuk memperbarui bobot parameter jaringan.
- **Validation Set (10%)**: Digunakan secara ketat untuk penyesuaian learning rate & kriteria *early stopping / best model checkpointing*.
- **Test Set (10%)**: Himpunan uji murni yang belum pernah dilihat sama sekali, dievaluasi tepat 1 kali setelah pelatihan tuntas.

In [ ]:
import io
import os
import tarfile
import warnings
import urllib.request
import numpy as np
import pandas as pd
from PIL import Image
import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Proteksi DecompressionBomb pada citra skala ultra-besar
Image.MAX_IMAGE_PIXELS = None
warnings.simplefilter('ignore', Image.DecompressionBombWarning)

VALID_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff')
SEED = 42

# Konfigurasi Jumlah Sampel per Dataset
XIZ9_SAMPLES_PER_CLASS = 3000        # 3.000 Real & 3.000 AI dari Xiz9Dataset (FLUX/MJ/SD/DALL-E)
SEMITRUTHS_SAMPLES_PER_CLASS = 2000  # 2.000 Real & 2.000 AI-Edited dari Semi-Truths (Inpainting/P2P)

# -------------------------------------------------------------------------
# A. Modul Pemuat Dataset KarmaLeo/Xiz9Dataset (AI-Generated SOTA vs Real)
# -------------------------------------------------------------------------
def load_xiz9_dataset(n_samples_per_class=3000):
    print("[*] Menyiapkan KarmaLeo/Xiz9Dataset (AI-Generated: FLUX.1, Midjourney, DALL-E, SD vs Real)...")
    CACHE_DIR = 'data/xiz9_cache'
    os.makedirs(f'{CACHE_DIR}/real', exist_ok=True)
    os.makedirs(f'{CACHE_DIR}/ai', exist_ok=True)

    existing_real = [os.path.join(f'{CACHE_DIR}/real', f) for f in os.listdir(f'{CACHE_DIR}/real') if f.lower().endswith(VALID_EXTS)]
    existing_ai = [os.path.join(f'{CACHE_DIR}/ai', f) for f in os.listdir(f'{CACHE_DIR}/ai') if f.lower().endswith(VALID_EXTS)]

    records = []
    for p in existing_real[:n_samples_per_class]:
        records.append({'image_path': p, 'label': 1, 'label_name': 'Real Image', 'manipulation': 'Authentic Photo/Art', 'source': 'Xiz9Dataset'})
    for p in existing_ai[:n_samples_per_class]:
        records.append({'image_path': p, 'label': 0, 'label_name': 'AI-Generated', 'manipulation': 'AI-Generated (FLUX/MJ/SD/DALL-E)', 'source': 'Xiz9Dataset'})

    count_real = len(existing_real)
    count_fake = len(existing_ai)
    target_real = n_samples_per_class
    target_fake = n_samples_per_class

    if count_real >= target_real and count_fake >= target_fake:
        print(f"[+] Menggunakan cache disk lokal Xiz9: {len(existing_real)} Real, {len(existing_ai)} AI.")
        return records

    print("[*] Mengunduh & memproses shard Parquet KarmaLeo/Xiz9Dataset secara langsung (bypass Arrow streaming)...")
    try:
        api = HfApi()
        repo_files = api.list_repo_files('KarmaLeo/Xiz9Dataset', repo_type='dataset')
        shards = sorted([f for f in repo_files if f.endswith('.parquet')])
    except Exception as e:
        print(f"[!] Info HfApi ({e}), menggunakan urutan shard standar data/...")
        shards = [f"data/test-{i:05d}.parquet" for i in range(10)] + [f"data/train-{i:05d}.parquet" for i in range(34)]

    for shard_path in shards:
        if count_real >= target_real and count_fake >= target_fake:
            break
        print(f"[*] Mengambil shard data: {shard_path}...")
        try:
            local_pq = hf_hub_download(repo_id='KarmaLeo/Xiz9Dataset', filename=shard_path, repo_type='dataset')
            pf = pq.ParquetFile(local_pq)

            for batch in pf.iter_batches(batch_size=256):
                if count_real >= target_real and count_fake >= target_fake:
                    break
                b_dict = batch.to_pydict()
                labels = b_dict.get('label', b_dict.get('labels', []))
                images = b_dict.get('image', b_dict.get('images', []))
                generators = b_dict.get('generator', [None] * len(labels))

                for idx, (lbl_val, img_data) in enumerate(zip(labels, images)):
                    if count_real >= target_real and count_fake >= target_fake:
                        break
                    lbl_str = str(lbl_val).lower().strip()
                    is_real = (lbl_val == 0) or ('real' in lbl_str) or (lbl_str == '0')

                    if is_real and count_real >= target_real:
                        continue
                    if (not is_real) and count_fake >= target_fake:
                        continue

                    try:
                        if isinstance(img_data, Image.Image):
                            img = img_data.convert('RGB')
                        elif isinstance(img_data, dict) and 'bytes' in img_data and img_data['bytes'] is not None:
                            img = Image.open(io.BytesIO(img_data['bytes'])).convert('RGB')
                        elif isinstance(img_data, (bytes, bytearray)):
                            img = Image.open(io.BytesIO(img_data)).convert('RGB')
                        else:
                            continue
                    except Exception:
                        continue

                    # Cegah RAM bloat: jika resolusi citra terlalu raksasa (> 2048px), resize dengan proporsional
                    if max(img.size) > 2048:
                        img.thumbnail((2048, 2048), Image.Resampling.BILINEAR)

                    gen_name = generators[idx] if idx < len(generators) and generators[idx] else 'FLUX/MJ/SD/DALL-E'

                    if is_real:
                        save_p = f"{CACHE_DIR}/real/xiz9_real_{count_real}.jpg"
                        if not os.path.exists(save_p):
                            img.save(save_p, 'JPEG', quality=95)
                        records.append({'image_path': save_p, 'label': 1, 'label_name': 'Real Image', 'manipulation': 'Authentic Photo/Art', 'source': 'Xiz9Dataset'})
                        count_real += 1
                    else:
                        save_p = f"{CACHE_DIR}/ai/xiz9_ai_{count_fake}.jpg"
                        if not os.path.exists(save_p):
                            img.save(save_p, 'JPEG', quality=95)
                        records.append({'image_path': save_p, 'label': 0, 'label_name': 'AI-Generated', 'manipulation': f'AI-Generated ({gen_name})', 'source': 'Xiz9Dataset'})
                        count_fake += 1

            # Bersihkan file shard lokal setelah diekstrak agar tidak membebani disk Colab
            try:
                if os.path.exists(local_pq):
                    os.remove(local_pq)
            except Exception:
                pass
        except Exception as err:
            print(f"[!] Error saat memproses shard {shard_path}: {err}")
            continue

    print(f"[+] Xiz9Dataset Siap: {count_real} Real | {count_fake} AI-Generated")
    return records

# -------------------------------------------------------------------------
# B. Modul Pemuat Dataset semi-truths/Semi-Truths (AI-Edited: Inpainting & P2P)
# -------------------------------------------------------------------------
def load_semitruths_dataset(n_samples_per_class=2000):
    print("[*] Menyiapkan Dataset semi-truths/Semi-Truths (AI-Edited: Inpainting & Prompt-to-Prompt)...")
    CACHE_DIR = 'data/semitruths_cache'
    os.makedirs(f'{CACHE_DIR}/real', exist_ok=True)
    os.makedirs(f'{CACHE_DIR}/edited', exist_ok=True)

    existing_real = [os.path.join(f'{CACHE_DIR}/real', f) for f in os.listdir(f'{CACHE_DIR}/real') if f.lower().endswith(VALID_EXTS)]
    existing_edited = [os.path.join(f'{CACHE_DIR}/edited', f) for f in os.listdir(f'{CACHE_DIR}/edited') if f.lower().endswith(VALID_EXTS)]

    if len(existing_real) >= n_samples_per_class and len(existing_edited) >= n_samples_per_class:
        print(f"[+] Menggunakan cache disk lokal Semi-Truths: {len(existing_real)} Real, {len(existing_edited)} AI Edited.")
        records = []
        for p in existing_real[:n_samples_per_class]:
            records.append({'image_path': p, 'label': 1, 'label_name': 'Real Image', 'manipulation': 'Authentic Scene', 'source': 'Semi-Truths'})
        for p in existing_edited[:n_samples_per_class]:
            records.append({'image_path': p, 'label': 0, 'label_name': 'AI-Generated', 'manipulation': 'AI-Edited (Inpainting/P2P)', 'source': 'Semi-Truths'})
        return records

    if len(existing_real) < n_samples_per_class:
        real_tar = f"{CACHE_DIR}/real_ade20k.tar.bz2"
        if not os.path.exists(real_tar):
            print("[*] Mengunduh citra Real Semi-Truths (ADE20K ~156 MB)...")
            urllib.request.urlretrieve('https://huggingface.co/datasets/semi-truths/Semi-Truths-Evalset/resolve/main/original/images/ADE20K.tar.bz2', real_tar)
        print("[*] Mengekstrak citra Real Semi-Truths...")
        with tarfile.open(real_tar, "r:bz2") as tar:
            count = len(existing_real)
            for m in tqdm(tar.getmembers(), desc="Ekstrak Real Semi-Truths"):
                if m.isfile() and m.name.lower().endswith(VALID_EXTS):
                    base_name = f"st_real_{count}_{os.path.basename(m.name)}"
                    dst = os.path.join(f"{CACHE_DIR}/real", base_name)
                    if not os.path.exists(dst):
                        f = tar.extractfile(m)
                        if f:
                            with open(dst, "wb") as out_f:
                                out_f.write(f.read())
                    count += 1
                    if count >= n_samples_per_class:
                        break
        try:
            if os.path.exists(real_tar):
                os.remove(real_tar)
        except Exception:
            pass

    if len(existing_edited) < n_samples_per_class:
        p2p_tar = f"{CACHE_DIR}/p2p_ade20k.tar.bz2"
        if not os.path.exists(p2p_tar):
            print("[*] Mengunduh citra AI Edited Semi-Truths (Prompt-to-Prompt ~130 MB)...")
            urllib.request.urlretrieve('https://huggingface.co/datasets/semi-truths/Semi-Truths-Evalset/resolve/main/p2p/ADE20K.tar.bz2', p2p_tar)
        print("[*] Mengekstrak citra AI Edited Semi-Truths...")
        with tarfile.open(p2p_tar, "r:bz2") as tar:
            count = len(existing_edited)
            for m in tqdm(tar.getmembers(), desc="Ekstrak AI-Edited Semi-Truths"):
                if m.isfile() and m.name.lower().endswith(VALID_EXTS):
                    base_name = f"st_p2p_{count}_{os.path.basename(m.name)}"
                    dst = os.path.join(f"{CACHE_DIR}/edited", base_name)
                    if not os.path.exists(dst):
                        f = tar.extractfile(m)
                        if f:
                            with open(dst, "wb") as out_f:
                                out_f.write(f.read())
                    count += 1
                    if count >= n_samples_per_class:
                        break
        try:
            if os.path.exists(p2p_tar):
                os.remove(p2p_tar)
        except Exception:
            pass

    all_real = [os.path.join(f'{CACHE_DIR}/real', f) for f in os.listdir(f'{CACHE_DIR}/real') if f.lower().endswith(VALID_EXTS)]
    all_edited = [os.path.join(f'{CACHE_DIR}/edited', f) for f in os.listdir(f'{CACHE_DIR}/edited') if f.lower().endswith(VALID_EXTS)]

    records = []
    for p in all_real[:n_samples_per_class]:
        records.append({'image_path': p, 'label': 1, 'label_name': 'Real Image', 'manipulation': 'Authentic Scene', 'source': 'Semi-Truths'})
    for p in all_edited[:n_samples_per_class]:
        records.append({'image_path': p, 'label': 0, 'label_name': 'AI-Generated', 'manipulation': 'AI-Edited (Inpainting/P2P)', 'source': 'Semi-Truths'})

    print(f"[+] Semi-Truths Siap: {len(all_real[:n_samples_per_class])} Real | {len(all_edited[:n_samples_per_class])} AI-Edited")
    return records

# -------------------------------------------------------------------------
# C. Penggabungan & Pembagian Data (Stratified 3-Way Split: 80:10:10)
# -------------------------------------------------------------------------
loaded_records = []
loaded_records.extend(load_xiz9_dataset(n_samples_per_class=XIZ9_SAMPLES_PER_CLASS))
loaded_records.extend(load_semitruths_dataset(n_samples_per_class=SEMITRUTHS_SAMPLES_PER_CLASS))

df_all = pd.DataFrame(loaded_records)
print("\n[*] Ringkasan Komposisi Dataset Mentah:")
print(df_all.groupby(['source', 'label_name']).size())

# Penyeimbangan Sampel (Seimbang Antara AI [0] dan Real [1])
df_ai = df_all[df_all['label'] == 0]
df_real = df_all[df_all['label'] == 1]
n_target = min(len(df_ai), len(df_real))

df_ai_sampled = df_ai.sample(n=n_target, random_state=SEED)
df_real_sampled = df_real.sample(n=n_target, random_state=SEED)
df_dataset = pd.concat([df_ai_sampled, df_real_sampled]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Split 1: 80% Train vs 20% (Val + Test)
train_df, val_test_df = train_test_split(
    df_dataset,
    test_size=0.20,
    random_state=SEED,
    stratify=df_dataset['label']
)

# Split 2: 10% Val vs 10% Test
val_df, test_df = train_test_split(
    val_test_df,
    test_size=0.50,
    random_state=SEED,
    stratify=val_test_df['label']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print('=' * 70)
print(f"Total Citra Keseluruhan  : {len(df_dataset):,} citra (XIZ9 + Semi-Truths)")
print(f"├─ Data Latih (Train 80%): {len(train_df):,} citra")
print(f"├─ Data Validasi (Val 10%): {len(val_df):,} citra (seleksi checkpoint terbaik)")
print(f"└─ Data Uji Murni (Test 10%): {len(test_df):,} citra (evaluasi final tak bias)")
print(f"\nRincian Distribusi Label:")
print(f"- AI-Generated (0) : {(df_dataset['label'] == 0).sum()} citra")
print(f"- Real Image (1)   : {(df_dataset['label'] == 1).sum()} citra")
print('=' * 70)


## 🛠️ 2. Pipeline Augmentasi Forensik (Native-Resolution & Anti-Kompresi)

In [ ]:
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

class RandomJPEGCompression:
    """
    Simulasi kompresi lossy JPEG acak pada skala resolusi asli.
    Melatih model agar kebal terhadap degradasi media sosial (WhatsApp, WebP, dll.).
    """
    def __init__(self, quality_range=(60, 95), p=0.45):
        self.quality_range = quality_range
        self.p = p

    def __call__(self, img):
        if random.random() < self.p:
            quality = random.randint(*self.quality_range)
            buf = BytesIO()
            img.save(buf, format='JPEG', quality=quality)
            buf.seek(0)
            return Image.open(buf).convert('RGB')
        return img


class NativeResolutionCrop:
    """
    Penyempurnaan V4: Menghindari bilinear downsampling global yang merusak sinyal frekuensi tinggi.
    Melakukan crop acak langsung pada skala piksel asli citra.
    """
    def __init__(self, size=224, is_train=True):
        self.size = size
        self.is_train = is_train

    def __call__(self, img):
        w, h = img.size
        if min(w, h) < self.size:
            scale = float(self.size) / float(min(w, h))
            new_w, new_h = int(round(w * scale)), int(round(h * scale))
            img = img.resize((new_w, new_h), Image.Resampling.BILINEAR)
            w, h = img.size

        if self.is_train:
            left = random.randint(0, w - self.size)
            top = random.randint(0, h - self.size)
        else:
            left = (w - self.size) // 2
            top = (h - self.size) // 2

        return img.crop((left, top, left + self.size, top + self.size))


class ForensicDataset(Dataset):
    """
    Dataset forensik yang mendukung data frame lokal maupun objek dataset biner.
    Membaca citra dari cache disk lokal dengan fallback proteksi format.
    """
    def __init__(self, data_source, transform=None):
        if isinstance(data_source, pd.DataFrame):
            self.records = data_source.to_dict('records')
        else:
            self.records = data_source
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]
        label = int(item["label"])

        if "image_path" in item and item["image_path"]:
            try:
                image = Image.open(item["image_path"]).convert("RGB")
            except Exception:
                image = Image.new("RGB", (224, 224), (128, 128, 128))
        elif "image" in item and item["image"] is not None:
            image = item["image"]
            if not isinstance(image, Image.Image):
                image = Image.open(BytesIO(image)).convert("RGB")
            else:
                image = image.convert("RGB")
        else:
            image = Image.new("RGB", (224, 224), (128, 128, 128))

        if self.transform:
            image = self.transform(image)

        return image, label

# Alias kompatibilitas ke belakang
HFToPyTorchDataset = ForensicDataset

# Definisi Transformasi
train_transforms = transforms.Compose([
    RandomJPEGCompression(quality_range=(60, 95), p=0.45),
    NativeResolutionCrop(size=224, is_train=True),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.4))], p=0.15),
    transforms.ColorJitter(brightness=0.06, contrast=0.06, saturation=0.06),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transforms = transforms.Compose([
    NativeResolutionCrop(size=224, is_train=False),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

BATCH_SIZE = 64  # Optimal untuk GPU Colab (T4 / L4 / A100)
use_pin = torch.cuda.is_available()

train_ds = ForensicDataset(train_df, transform=train_transforms)
val_ds = ForensicDataset(val_df, transform=eval_transforms)
test_ds = ForensicDataset(test_df, transform=eval_transforms)

num_w = 2 if os.name != 'nt' else 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_w, pin_memory=use_pin, persistent_workers=(num_w > 0))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_w, pin_memory=use_pin, persistent_workers=(num_w > 0))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_w, pin_memory=use_pin, persistent_workers=(num_w > 0))

print(f"✅ DataLoader Berhasil Dikonfigurasi (Batch Size: {BATCH_SIZE})")
print(f"   - Batch Train : {len(train_loader):,}")
print(f"   - Batch Val   : {len(val_loader):,}")
print(f"   - Batch Test  : {len(test_loader):,}")


## 🧠 3. Definisi Arsitektur Lengkap AIForensicNet V4 (Self-Contained)
Implementasi modular mandiri yang mencakup:
1. **NPRModule** (Neighboring Pixel Relationships - CVPR 2024)
2. **FFTSpectralBranch** (Fast Fourier Transform 2D Spectral Feature Extraction)
3. **MultiScaleForensicFilterBankV4** (15-Channel Spatial Forensic Signals)
4. **ForensicBlock** (Depthwise 7x7 + Squeeze-Excitation Residual Block)
5. **MultiStatPool2d** (Global Mean + Standard Deviation Pooling)
6. **AIForensicNetV4** (Tri-Stream Deep Neural Network)

In [ ]:
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced = max(8, channels // reduction)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, reduced, bias=False),
            nn.GELU(),
            nn.Linear(reduced, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.fc(x).view(b, c, 1, 1)
        return x * w


class ForensicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.dwconv = nn.Conv2d(in_c, in_c, kernel_size=7, stride=stride, padding=3, groups=in_c, bias=False)
        self.norm = nn.BatchNorm2d(in_c)
        self.pwconv1 = nn.Conv2d(in_c, out_c * 2, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(out_c * 2, out_c, kernel_size=1)
        self.se = SqueezeExcitation(out_c)
        
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.dwconv(x)
        out = self.norm(out)
        out = self.pwconv1(out)
        out = self.act(out)
        out = self.pwconv2(out)
        out = self.se(out)
        return out + res


class NPRModule(nn.Module):
    def forward(self, x):
        down = F.interpolate(x, scale_factor=0.5, mode='nearest')
        up = F.interpolate(down, size=x.shape[2:], mode='nearest')
        return x - up


class FFTSpectralBranch(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=3, padding=1, bias=False)
        self.norm1 = nn.BatchNorm2d(16)
        self.act1 = nn.GELU()
        self.conv2 = nn.Conv2d(16, out_channels, kernel_size=3, stride=2, padding=1, bias=False)
        self.norm2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.GELU()

    def forward(self, x):
        fft = torch.fft.rfft2(x, norm="ortho")
        mag = torch.log(torch.abs(fft) + 1e-6)
        mag_interp = F.interpolate(mag, size=x.shape[2:], mode='bilinear', align_corners=False)
        feat = self.act1(self.norm1(self.conv1(mag_interp)))
        feat = self.act2(self.norm2(self.conv2(feat)))
        return feat


class MultiScaleForensicFilterBankV4(nn.Module):
    def __init__(self):
        super().__init__()
        lap3 = torch.tensor([[-1., -1., -1.], [-1., 8., -1.], [-1., -1., -1.]], dtype=torch.float32)
        lap5 = torch.tensor([[0., 0., -1., 0., 0.], [0., -1., -2., -1., 0.], [-1., -2., 16., -2., -1.], [0., -1., -2., -1., 0.], [0., 0., -1., 0., 0.]], dtype=torch.float32)
        dog = torch.tensor([[-1., -3., -4., -3., -1.], [-3., 0., 6., 0., -3.], [-4., 6., 20., 6., -4.], [-3., 0., 6., 0., -3.], [-1., -3., -4., -3., -1.]], dtype=torch.float32) / 16.0
        sobel_h = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], dtype=torch.float32)
        sobel_v = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], dtype=torch.float32)

        k3 = torch.stack([lap3, sobel_h, sobel_v]).unsqueeze(1).repeat(3, 1, 1, 1)
        k5 = torch.stack([lap5, dog]).unsqueeze(1).repeat(3, 1, 1, 1)

        self.register_buffer('k3', k3)
        self.register_buffer('k5', k5)
        self.npr = NPRModule()

    def forward(self, x):
        f3 = F.conv2d(x, self.k3, padding=1, groups=3)
        f5 = F.conv2d(x, self.k5, padding=2, groups=3)
        sh = f3[:, 1::3, :, :]
        sv = f3[:, 2::3, :, :]
        smag = torch.sqrt(sh ** 2 + sv ** 2 + 1e-6)
        lap3_out = f3[:, 0::3, :, :]
        lap5_out = f5[:, 0::2, :, :]
        dog_out = f5[:, 1::2, :, :]
        npr_out = self.npr(x)
        return torch.cat([lap3_out, lap5_out, dog_out, smag, npr_out], dim=1)


class MultiStatPool2d(nn.Module):
    def forward(self, x):
        mean = torch.mean(x, dim=(2, 3))
        var = torch.var(x, dim=(2, 3), unbiased=False)
        std = torch.sqrt(var + 1e-6)
        return torch.cat([mean, std], dim=1)


class AIForensicNetV4(nn.Module):
    def __init__(self, num_classes=2, dropout_p=0.25):
        super().__init__()
        self.forensic_bank = MultiScaleForensicFilterBankV4()
        self.forensic_stem = nn.Sequential(
            nn.Conv2d(15, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU()
        )
        self.spectral_branch = FFTSpectralBranch(in_channels=3, out_channels=32)
        self.rgb_stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU()
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(96, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU()
        )
        self.stage1 = nn.Sequential(ForensicBlock(64, 64), ForensicBlock(64, 64))
        self.stage2 = nn.Sequential(ForensicBlock(64, 128, stride=2), ForensicBlock(128, 128))
        self.stage3 = nn.Sequential(ForensicBlock(128, 256, stride=2), ForensicBlock(256, 256))
        self.stage4 = nn.Sequential(ForensicBlock(256, 512, stride=2), ForensicBlock(512, 512))
        self.pool = MultiStatPool2d()
        self.head = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(512 * 2, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(p=0.15),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        feat_forensic = self.forensic_stem(self.forensic_bank(x))
        feat_spectral = self.spectral_branch(x)
        feat_rgb = self.rgb_stem(x)
        tri_feat = torch.cat([feat_forensic, feat_spectral, feat_rgb], dim=1)
        out = self.fusion(tri_feat)
        out = self.stage1(out)
        out = self.stage2(out)
        out = self.stage3(out)
        out = self.stage4(out)
        pooled = self.pool(out)
        return self.head(pooled)

# Inisialisasi Model
model = AIForensicNetV4(num_classes=2, dropout_p=0.25).to(device)
total_params = sum(p.numel() for p in model.parameters())
print('=' * 70)
print(f"🧠 Model AIForensicNet V4 Siap: {total_params:,} Parameter (~{total_params/1e6:.2f}M)")
print('=' * 70)

## 🚀 4. Eksekusi Loop Pelatihan & Validasi Teratur (15 Epoch)
- **Checkpoint Selection**: Bobot model terbaik disimpan hanya saat skor **Validation Macro F1** mencapai titik tertinggi baru.
- **AMP FP16**: Akselerasi kecepatan hingga 2x lipat dan hemat VRAM.

In [ ]:
EPOCHS = 15
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-2
LABEL_SMOOTHING = 0.05
MODEL_SAVE_PATH = 'best_aiforensicnet_v4_ai_vs_real.pth'

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
eval_criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

warmup_epochs = 2
cosine_epochs = EPOCHS - warmup_epochs

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return float(epoch + 1) / float(max(1, warmup_epochs))
    else:
        progress = float(epoch - warmup_epochs) / float(max(1, cosine_epochs))
        return max(1e-3, 0.5 * (1.0 + np.cos(np.pi * progress)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

best_val_f1 = 0.0
best_val_acc = 0.0
best_weights = copy.deepcopy(model.state_dict())

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

print(f"Mulai Pelatihan AIForensicNet V4 ({EPOCHS} Epoch, AMP FP16)...")
for epoch in range(EPOCHS):
    start_time = time.time()
    current_lr = optimizer.param_groups[0]['lr']

    # ---------------- FASE TRAINING ----------------
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1:02d}/{EPOCHS}] Train (LR: {current_lr:.1e})", leave=True)
    optimizer.zero_grad(set_to_none=True)

    for images, labels in train_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels).item()
        train_total += images.size(0)
        train_bar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{(train_correct / train_total) * 100:.2f}%"})

    scheduler.step()
    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total

    # ---------------- FASE VALIDASI ----------------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    all_val_preds = []
    all_val_labels = []

    val_bar = tqdm(val_loader, desc=f"Epoch [{epoch+1:02d}/{EPOCHS}] Val  ", leave=False)
    with torch.no_grad():
        for images, labels in val_bar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss = eval_criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += images.size(0)
            all_val_preds.extend(preds.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    epoch_val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')
    elapsed_sec = time.time() - start_time

    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    history['val_f1'].append(epoch_val_f1)

    print(f"📊 [Epoch {epoch+1:02d}/{EPOCHS} - {elapsed_sec/60:.2f} m] "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% || "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}% | Val Macro F1: {epoch_val_f1:.4f}")

    if epoch_val_f1 > best_val_f1:
        best_val_f1 = epoch_val_f1
        best_val_acc = epoch_val_acc
        best_weights = copy.deepcopy(model.state_dict())
        torch.save(best_weights, MODEL_SAVE_PATH)
        print(f"   🌟 CHECKPOINT TERBAIK DISIMPAN: '{MODEL_SAVE_PATH}' (Val F1: {best_val_f1:.4f} | Acc: {best_val_acc*100:.2f}%)")
    print('-' * 70)

## 📈 5. Visualisasi Kurva Loss, Akurasi, dan F1-Score

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(18, 5))

# Plot Loss
plt.subplot(1, 3, 1)
plt.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
plt.plot(epochs_range, history['val_loss'], 'r-s', label='Val Loss')
plt.title('Kurva Loss Pelatihan & Validasi (V4)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Plot Akurasi
plt.subplot(1, 3, 2)
plt.plot(epochs_range, [x * 100 for x in history['train_acc']], 'b-o', label='Train Acc')
plt.plot(epochs_range, [x * 100 for x in history['val_acc']], 'g-^', label='Val Acc')
plt.title('Kurva Akurasi (%) per Epoch (V4)')
plt.xlabel('Epoch')
plt.ylabel('Akurasi (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Plot F1-Score
plt.subplot(1, 3, 3)
plt.plot(epochs_range, history['val_f1'], 'm-d', label='Val Macro F1')
plt.title('Kurva Validation Macro F1-Score (V4)')
plt.xlabel('Epoch')
plt.ylabel('F1 Score')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 🏆 6. Evaluasi Akhir pada HELD-OUT TEST SET (10% Data Independen)
Tahap pembuktian ilmiah: Model terbaik dimuat kembali dan dievaluasi terhadap himpunan uji yang **sama sekali belum pernah dilihat** selama proses pelatihan maupun pemilihan checkpoint.

In [ ]:
print("Memuat bobot terbaik untuk pengujian independen...")
model.load_state_dict(best_weights)
model.eval()

test_preds = []
test_probs = []
test_targets = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluasi Test Set"):
        images = images.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        test_preds.extend(preds.cpu().numpy())
        test_probs.extend(probs[:, 1].cpu().numpy())  # Probabilitas Real Image (1)
        test_targets.extend(labels.numpy())

test_targets = np.array(test_targets)
test_preds = np.array(test_preds)
test_probs = np.array(test_probs)

test_acc = (test_preds == test_targets).mean() * 100.0
test_f1 = f1_score(test_targets, test_preds, average='macro')
test_auc = roc_auc_score(test_targets, test_probs) * 100.0

print('=' * 70)
print(f"🎯 Akurasi Himpunan Uji (Test Accuracy) : {test_acc:.2f}%")
print(f"🎯 Macro F1-Score Himpunan Uji          : {test_f1:.4f}")
print(f"🎯 ROC-AUC Score Himpunan Uji           : {test_auc:.2f}%")
print('=' * 70)

print("\n📋 CLASSIFICATION REPORT (AIForensicNet V4 pada Test Set Murni):")
print(classification_report(test_targets, test_preds, target_names=["AI-Generated (0)", "Real Image (1)"], digits=4))

# Visualisasi Confusion Matrix
cm = confusion_matrix(test_targets, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["AI-Generated", "Real Image"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Confusion Matrix - AIForensicNet V4 (Held-Out Test Set)')
plt.show()

## 🔍 7. Fungsi Inferensi Prediksi Citra Tunggal
Fungsi cepat untuk menguji citra tunggal (misal foto kiriman atau dokumen baru).

In [ ]:
def predict_single_image(image_path, model, device):
    if not os.path.exists(image_path):
        print(f"File tidak ditemukan: {image_path}")
        return

    image = Image.open(image_path)
    if image.mode == 'P' and 'transparency' in image.info:
        image = image.convert('RGBA')
    if image.mode != 'RGB':
        image = image.convert('RGB')

    tensor = eval_transforms(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            outputs = model(tensor)
        probs = torch.softmax(outputs, dim=1)[0]
        ai_prob = probs[0].item() * 100.0
        real_prob = probs[1].item() * 100.0
        pred_idx = torch.argmax(probs).item()

        if ai_prob >= 80.0:
            verdict = 'AI-Generated Image'
            status = 'Tingkat Keyakinan Tinggi (AI >= 80%)'
            conf = ai_prob
        elif real_prob >= 80.0:
            verdict = 'Real Image'
            status = 'Tingkat Keyakinan Tinggi (Real >= 80%)'
            conf = real_prob
        else:
            verdict = 'Gambar ini terdeteksi sebagian diedit atau dimodifikasi menggunakan kecerdasan buatan (AI)'
            status = 'Tingkat Keyakinan Rendah (< 80%)'
            conf = max(ai_prob, real_prob)

        print('=' * 65)
        print(f"File Uji          : {os.path.basename(image_path)}")
        print(f"Status Keyakinan  : {status}")
        print(f"Hasil Prediksi    : {verdict}")
        print(f"Tingkat Keyakinan : {conf:.2f}%")
        print(f"  ├─ Probabilitas AI   : {ai_prob:.2f}%")
        print(f"  └─ Probabilitas Real : {real_prob:.2f}%")
        print('=' * 65)

# Contoh pengujian (jika ada file sampel di direktori):
for sample_file in ['test1.webp', 'real.jpg', 'wa.jpeg']:
    if os.path.exists(sample_file):
        predict_single_image(sample_file, model, device)